# 05 — Image Inference + SAFE/UNSAFE Classification
## SmartMine Vision AI · Stage 1: PPE Detection

---

### Objectives

1. Run the trained YOLOv8 model on **random test images**.
2. Visualise **bounding boxes, class labels, and confidence scores**.
3. Apply the **SAFE / UNSAFE** logic from `ppe_classifier.py`.
4. Save annotated results to `outputs/images/`.

## 1. Setup

In [ ]:
import sys, random
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path().resolve().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

from src.ppe_detection.utils import MODELS_DIR, TEST_IMAGES, OUTPUTS_DIR, ensure_dirs
from src.ppe_detection.inference import load_model, run_image_inference, draw_detections
from src.ppe_detection.ppe_classifier import classify_workers, compliance_color

ensure_dirs()

WEIGHTS = MODELS_DIR / "yolov8n_ppe_baseline.pt"
model = load_model(WEIGHTS)
print(f"Model loaded: {WEIGHTS}")

## 2. Run Inference on Random Test Images

In [ ]:
random.seed(0)
test_images = list(TEST_IMAGES.glob("*.jpg"))
samples = random.sample(test_images, min(6, len(test_images)))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, img_path in zip(axes, samples):
    annotated, detections = run_image_inference(model, img_path, conf=0.4, save=True)
    ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    ax.set_title(img_path.name[:40], fontsize=8)
    ax.axis("off")

plt.suptitle("Test Set Inference Results", fontsize=14)
plt.tight_layout()
grid_path = OUTPUTS_DIR / "images" / "inference_grid.png"
plt.savefig(str(grid_path), dpi=150)
plt.show()
print(f"Grid saved → {grid_path}")

## 3. SAFE / UNSAFE Classification

### How it works

The `classify_workers()` function:
1. Identifies every **Person** bounding box.
2. Checks if a **Hardhat** and **Safety Vest** overlap with that person's box (using IoU).
3. Also checks for explicit violation classes (`NO-Hardhat`, `NO-Safety Vest`).
4. Labels the worker **SAFE** (green) or **UNSAFE** (red).

In [ ]:
def overlay_compliance(
    image: np.ndarray,
    detections,
    iou_thresh: float = 0.05,
) -> np.ndarray:
    """Draw SAFE/UNSAFE labels on top of detection bounding boxes."""
    canvas = draw_detections(image, detections)
    workers = classify_workers(detections, iou_thresh=iou_thresh)
    for w in workers:
        x1, y1, x2, y2 = w.person_bbox
        color = compliance_color(w.status)
        label = w.status.value
        cv2.rectangle(canvas, (x1, y1 - 3), (x2, y1 - 25), color, -1)
        cv2.putText(canvas, label, (x1 + 4, y1 - 6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 0, 0), 2)
    return canvas


# Apply to the same samples
from src.ppe_detection.inference import predict_image

fig2, axes2 = plt.subplots(2, 3, figsize=(18, 10))
axes2 = axes2.flatten()

for ax, img_path in zip(axes2, samples):
    img = cv2.imread(str(img_path))
    detections = predict_image(model, img, conf=0.4)
    canvas = overlay_compliance(img, detections)
    ax.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
    ax.set_title(img_path.name[:40], fontsize=8)
    ax.axis("off")

plt.suptitle("PPE Compliance: SAFE (green) / UNSAFE (red)", fontsize=14)
plt.tight_layout()
compliance_path = OUTPUTS_DIR / "images" / "compliance_grid.png"
plt.savefig(str(compliance_path), dpi=150)
plt.show()
print(f"Saved → {compliance_path}")

## 4. Single Image Deep Dive

In [ ]:
# Change img_path to any image you want to inspect
img_path = samples[0]
img = cv2.imread(str(img_path))
detections = predict_image(model, img, conf=0.4)
workers = classify_workers(detections)

print(f"Image: {img_path.name}")
print(f"Detections: {len(detections)}")
print(f"Workers found: {len(workers)}")
for i, w in enumerate(workers):
    print(f"  Worker {i+1}: {w.status.value} | hardhat={w.has_hardhat} | vest={w.has_vest}")
    if w.violations:
        print(f"    Violations: {', '.join(w.violations)}")

## 5. Conclusions & Next Steps

**What to check:**
- Are SAFE/UNSAFE labels visually correct?
- Are there false negatives (unsafe worker classified as SAFE)?
- Adjust `iou_thresh` in `classify_workers()` if needed.

**Next:** `06_video_inference.ipynb` — run the full pipeline on a video stream.